# CAPTION · Mô tả ảnh (CIDEr-D) - KHÔNG dùng trọng số pretrained

Sinh câu bằng CNN + LSTM train từ đầu trên 3500 ảnh sẽ rất yếu, nên dùng **truy hồi + đồng thuận** (nearest-neighbor captioning):
1. Train **ResNet-18 khởi tạo ngẫu nhiên** dự đoán ~400 "từ khái niệm" (danh từ/động từ hay gặp) có trong 5 caption của ảnh
   (multi-label, nhãn lấy từ chính caption của bài, không dùng dữ liệu ngoài).
2. Đặc trưng của CNN đó dùng để tìm `K` ảnh train giống ảnh cần mô tả nhất.
3. Gom caption của các ảnh láng giềng, chọn câu **đồng thuận nhất** (MBR: tổng độ tương đồng CIDEr với các câu khác,
   có trọng số theo độ giống của láng giềng).

Thư viện: `torch`, `torchvision` (chỉ kiến trúc), `numpy`, `pandas`, `Pillow`, `json`, `re`, `collections`, `math`, `matplotlib`.

In [ ]:
import os, json, re, math, time, random
from collections import Counter, defaultdict
import torch          # import trước pandas để tránh lỗi DLL trên Windows
import torch.nn as nn
import torch.nn.functional as F
import torchvision
from torchvision import transforms as T
import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

DATA = '.'
DEV = 'cuda' if torch.cuda.is_available() else 'cpu'
SEED, V, IMG, BS = 42, 400, 128, 64
BUDGET_FT = 10      # phút train CNN
T0 = time.time()
open('run_log.txt', 'w').close()


def log(*a):
    """In ra màn hình và ghi vào run_log.txt (xem được tiến độ khi chạy bằng nbconvert)."""
    msg = f'[{(time.time() - T0) / 60:5.1f} phút] ' + ' '.join(map(str, a)); print(msg)
    with open('run_log.txt', 'a', encoding='utf-8') as f: f.write(msg + '\n')


def seed_all(s=SEED):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

## 1. Dữ liệu + hàm chấm CIDEr-D (giống `scorer.py`)

In [ ]:
train_caps = json.load(open(f'{DATA}/train_captions.json', encoding='utf-8'))
val_caps = json.load(open(f'{DATA}/val_captions.json', encoding='utf-8'))
tr_ids, val_ids = sorted(train_caps), sorted(val_caps)
test_ids = sorted(f[:-4] for f in os.listdir(f'{DATA}/test'))
log('train/val/test:', len(tr_ids), len(val_ids), len(test_ids))


def toks(s): return re.findall(r"[a-z0-9']+", s.lower())
def ngrams(t, n): return Counter(tuple(t[i:i + n]) for i in range(len(t) - n + 1))


def make_df(ref_lists):
    df = Counter()
    for rs in ref_lists:
        df.update({g for r in rs for n in range(1, 5) for g in ngrams(r, n)})
    return df, math.log(float(len(ref_lists)))


def cider_vec(t, df, logN):
    vs, norms = [defaultdict(float) for _ in range(4)], [0.0] * 4
    for n in range(1, 5):
        for g, tf in ngrams(t, n).items():
            vs[n - 1][g] = tf * (logN - math.log(max(1.0, df.get(g, 0.0))))
            norms[n - 1] += vs[n - 1][g] ** 2
    return vs, [math.sqrt(x) for x in norms], len(t)


def cider_sim(a, b, sigma=6.0):
    (va, na, la), (vb, nb, lb) = a, b
    out = 0.0
    for n in range(4):
        s = sum(min(x, vb[n][g]) * vb[n][g] for g, x in va[n].items() if g in vb[n])
        if na[n] and nb[n]: s /= na[n] * nb[n]
        out += s * math.e ** (-((la - lb) ** 2) / (2 * sigma ** 2))
    return out / 4 * 10


def cider(preds, refs=val_caps):
    R = {i: [toks(r) for r in refs[i] if toks(r)] for i in preds}
    df, logN = make_df(R.values())
    return float(np.mean([np.mean([cider_sim(cider_vec(toks(preds[i]) or ['a'], df, logN), cider_vec(r, df, logN))
                                   for r in R[i]]) for i in preds]))

## 2. Từ vựng khái niệm (nhãn multi-label lấy từ caption train)

In [ ]:
STOP = set('a an the of on in at to and with is are its it this that for from by as up down near next while some there two three '
           'his her their other each one very top front side back out into over under has have be being his'.split())
img_words = {i: {w for c in train_caps[i] for w in toks(c) if w not in STOP and len(w) > 2} for i in tr_ids}
df_words = Counter(w for ws in img_words.values() for w in ws)
vocab = [w for w, c in df_words.most_common(V) if c >= 5]
w2i = {w: k for k, w in enumerate(vocab)}
Ytr = np.zeros((len(tr_ids), len(vocab)), np.float32)
for r, i in enumerate(tr_ids):
    for w in img_words[i]:
        if w in w2i: Ytr[r, w2i[w]] = 1
log(f'{len(vocab)} từ khái niệm, trung bình {Ytr.sum(1).mean():.1f} từ/ảnh; ví dụ: {vocab[:15]}')

In [ ]:
load = lambda folder, ids: [Image.open(f'{DATA}/{folder}/{i}.jpg').convert('RGB').resize((IMG + 16, IMG + 16), Image.BILINEAR)
                            for i in tqdm(ids, desc=folder, leave=False)]
Itr, Iva, Ite = load('train', tr_ids), load('val', val_ids), load('test', test_ids)
log('đã nạp ảnh')

## 3. Train ResNet-18 từ đầu (trong ngân sách thời gian)

In [ ]:
norm = T.Normalize([0.5, 0.5, 0.5], [0.25, 0.25, 0.25])
aug = T.Compose([T.RandomResizedCrop(IMG, scale=(0.6, 1.0)), T.RandomHorizontalFlip(),
                 T.ColorJitter(0.3, 0.3, 0.3, 0.03), T.ToTensor(), norm])
plain = T.Compose([T.CenterCrop(IMG), T.ToTensor(), norm])


class DS(torch.utils.data.Dataset):
    def __len__(self): return len(Itr)
    def __getitem__(self, i): return aug(Itr[i]), torch.from_numpy(Ytr[i])


def make_model():
    return torchvision.models.resnet18(weights=None, num_classes=len(vocab)).to(DEV, memory_format=torch.channels_last)


def train(budget_min, lr=2e-3, max_epochs=200):
    seed_all()
    m = make_model()
    dl = torch.utils.data.DataLoader(DS(), batch_size=BS, shuffle=True, num_workers=0, drop_last=True)
    opt = torch.optim.AdamW(m.parameters(), lr=lr, weight_decay=5e-4)
    crit = nn.BCEWithLogitsLoss()

    def step(x, y, update=True):
        x, y = x.to(DEV, memory_format=torch.channels_last), y.to(DEV)
        loss = crit(m(x), y)
        opt.zero_grad(set_to_none=True); loss.backward()
        if update: opt.step(); sched.step()
        return loss.item()

    m.train(); it = iter(dl); step(*next(it), update=False)
    t0 = time.time()
    for _ in range(3): step(*next(it), update=False)
    if DEV == 'cuda': torch.cuda.synchronize()
    sec = (time.time() - t0) / 3
    epochs = int(np.clip(budget_min * 60 / (sec * len(dl)), 2, max_epochs))
    log(f'{sec:.2f} s/bước × {len(dl)} bước/epoch, chạy {epochs} epoch ≈ {epochs * len(dl) * sec / 60:.1f} phút')
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=lr, total_steps=epochs * len(dl), pct_start=0.2)
    t_train = time.time()
    for ep in range(epochs):
        m.train(); tot = 0.0
        for x, y in dl: tot += step(x, y)
        assert np.isfinite(tot), 'loss NaN/inf'
        if (ep + 1) % max(1, epochs // 10) == 0 or ep + 1 == epochs:
            log(f'  epoch {ep + 1}/{epochs} loss {tot / len(dl):.4f} · còn ~{(time.time() - t_train) / (ep + 1) * (epochs - ep - 1) / 60:.1f} phút')
    return m


model = train(BUDGET_FT)
torch.save(model.state_dict(), 'caption_resnet18_scratch.pt')

## 4. Đặc trưng ảnh (lớp trước fc, trung bình ảnh gốc + ảnh lật) để tìm láng giềng

In [ ]:
@torch.no_grad()
def embed(ims, bs=128):
    model.eval(); body = nn.Sequential(*list(model.children())[:-1]); out, probs = [], []
    for i in range(0, len(ims), bs):
        x = torch.stack([plain(im) for im in ims[i:i + bs]]).to(DEV, memory_format=torch.channels_last)
        f = (body(x) + body(torch.flip(x, dims=[3]))).flatten(1) / 2
        out.append(F.normalize(f, dim=1).cpu()); probs.append(torch.sigmoid(model.fc(f)).cpu())
    return torch.cat(out).numpy(), torch.cat(probs).numpy()


Etr, _ = embed(Itr)
Eva, Pva = embed(Iva)
Ete, _ = embed(Ite)
top_words = [[vocab[k] for k in np.argsort(-p)[:6]] for p in Pva[:4]]
log('từ khái niệm dự đoán cho 4 ảnh val đầu:', top_words)

## 5. Truy hồi + đồng thuận (MBR), chọn `K` trên validation

In [ ]:
df_tr, logN_tr = make_df([[toks(c) for c in train_caps[i]] for i in tr_ids])
vec_tr = {i: [cider_vec(toks(c), df_tr, logN_tr) for c in train_caps[i]] for i in tr_ids}


def caption(E, ids, K, weighted=True):
    sims = E @ Etr.T
    out = {}
    for r, i in enumerate(tqdm(ids, leave=False)):
        nb = np.argsort(-sims[r])[:K]
        cand = [(train_caps[tr_ids[j]][c], vec_tr[tr_ids[j]][c], sims[r, j] if weighted else 1.0) for j in nb for c in range(5)]
        best, best_s = None, -1
        for a, (txt, va_, _) in enumerate(cand):
            s = sum(wb * cider_sim(va_, vb) for b, (_, vb, wb) in enumerate(cand) if b != a)
            if s > best_s: best, best_s = txt, s
        out[i] = best
    return out


rows = []
nn1 = {i: train_caps[tr_ids[np.argmax(Eva[r] @ Etr.T)]][0] for r, i in enumerate(val_ids)}
rows.append({'cách': 'láng giềng gần nhất, câu đầu tiên', 'val CIDEr-D': round(cider(nn1), 4)})
for K in [5, 10, 20]:
    rows.append({'cách': f'MBR trên {K} láng giềng', 'K': K, 'val CIDEr-D': round(cider(caption(Eva, val_ids, K)), 4)})
res = pd.DataFrame(rows); print(res.to_string(index=False))
best = res.dropna(subset=['K']).sort_values('val CIDEr-D', ascending=False).iloc[0]
K_BEST = int(best.K)
log(f'chọn K = {K_BEST} (val CIDEr-D {best["val CIDEr-D"]})')

So sánh: baseline truy hồi theo điểm ảnh thu nhỏ 16×16 của `starter.py`.

In [ ]:
px = lambda ims: np.stack([np.asarray(im.resize((16, 16), Image.BILINEAR), np.float32).ravel() / 255 for im in ims])
Ptr, Pv = px(Itr), px(Iva)
Ptr, Pv = Ptr - Ptr.mean(1, keepdims=True), Pv - Pv.mean(1, keepdims=True)
starter = {i: train_caps[tr_ids[np.argmin(((Ptr - Pv[r]) ** 2).sum(1))]][0] for r, i in enumerate(val_ids)}
log('starter val CIDEr-D:', round(cider(starter), 4))

## 6. Bản nộp

In [ ]:
p_test = caption(Ete, test_ids, K_BEST)
sub = pd.DataFrame({'id': test_ids, 'caption': [p_test[i] for i in test_ids]})
assert len(sub) == len(test_ids)
sub.to_csv('submission.csv', index=False, encoding='utf-8')
log('đã ghi submission.csv')
fig, ax = plt.subplots(2, 4, figsize=(16, 7.5))
for a, i in zip(ax.ravel(), test_ids[8:16]):
    a.imshow(Image.open(f'{DATA}/test/{i}.jpg')); a.set_title(p_test[i], fontsize=9, wrap=True); a.axis('off')
plt.tight_layout(); plt.show()